# Ансамбли моделей

Сравним одиночное дерево, случайный лес и градиентный бустинг через стратифицированную кросс-валидацию.

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

In [2]:
features, target = load_breast_cancer(return_X_y=True, as_frame=True)
validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, min_samples_leaf=2, n_jobs=1, random_state=42),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=42),
}

In [3]:
rows = []
for name, model in models.items():
    scores = cross_validate(
        model, features, target, cv=validation,
        scoring={"f1": "f1", "roc_auc": "roc_auc"}, n_jobs=1
    )
    rows.append({
        "model": name,
        "f1_mean": scores["test_f1"].mean(),
        "f1_std": scores["test_f1"].std(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
    })
results = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
results

,model,f1_mean,f1_std,roc_auc_mean
2,Gradient Boosting,0.967080,0.018796,0.992179
1,Random Forest,0.962599,0.012880,0.990369
0,Decision Tree,0.936434,0.011699,0.943656
